In [17]:
import mlflow
import os

# os.environ["AWS_PROFILE"] = "default" # fill in with your AWS profile. More info: https://docs.aws.amazon.com/sdk-for-java/latest/developer-guide/setup.html#setup-credentials

TRACKING_SERVER_HOST = "3.145.15.159" # fill in with the public DNS of the EC2 instance
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [18]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://3.145.15.159:5000'


In [19]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-21/1', creation_time=1756546682593, experiment_id='1', last_update_time=1756546682593, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-21/0', creation_time=1756546031456, experiment_id='0', last_update_time=1756546031456, lifecycle_stage='active', name='Default', tags={}>]

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2025/08/30 09:44:40 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp6w3espdu/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.0.2', 'cloudpickle==2.0.0']. Set logging level to DEBUG to see the full traceback. 
2025/08/30 09:44:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run fortunate-trout-727 at: http://3.145.15.159:5000/#/experiments/1/runs/025c043373014a7d93f2bec36bfd8c6f
🧪 View experiment at: http://3.145.15.159:5000/#/experiments/1


ProfileNotFound: The config profile (default) could not be found

In [22]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-21/1', creation_time=1756546682593, experiment_id='1', last_update_time=1756546682593, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-21/0', creation_time=1756546031456, experiment_id='0', last_update_time=1756546031456, lifecycle_stage='active', name='Default', tags={}>]

In [23]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [24]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1756546764725, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1756546764725, latest_versions=[], name='iris-classifier', tags={}>]

In [25]:
run_id = client.search_runs(experiment_ids=['1'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Registered model 'iris-classifier' already exists. Creating a new version of this model...


ProfileNotFound: The config profile (default) could not be found

In [26]:
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1756546764725, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1756546764725, latest_versions=[], name='iris-classifier', tags={}>]